In [1]:
# 0. 수집 전에 robots.txt 확인
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Referer": "https://www.yes24.com/Product/Category/MonthWeekBestSeller",
}

robots = requests.get("https://www.yes24.com/robots.txt", headers=HEADERS, timeout=30)
print(robots.status_code)
print(robots.text)

200
User-agent: *
Disallow: /*.pdf$
Disallow: /templates/
Disallow: /Templates/
Disallow: /member/
Disallow: /Member/
Disallow: /partnersso/
Disallow: /PartnerSSO/
Disallow: /modules/
Disallow: /Modules/
Disallow: /exchange/
Disallow: /Exchange/
Disallow: /common/
Disallow: /Common/
Disallow: /partner/
Disallow: /Partner/
Disallow: /goods/
Disallow: /Goods/
Disallow: /Cooperate/Yes24Gateway.aspx
Disallow: /Product/Base/
Disallow: /Product/AuthorFile/CommunityMessage/
Disallow: /product/goods/AuthorFile/CommunityMessage/
Disallow: /product/goods/AuthorFile/CommunityMessage/
Disallow: /product/goods/addModules/
Disallow: /product/goods/addModules/
Disallow: /product/goods/communityModules/
Disallow: /product/goods/communityModules/
Disallow: /Product/Goods/JsonGoodsList
Disallow: /product/goods/JsonGoodsList
Disallow: /product/goods/jsongoodslist
Disallow: /Product/Base/ExceptionNotice
Disallow: /Product/CommunityModules/GoodsReviewList
Disallow: /Product/CommunityModules/ReviewWeekList


**robots.txt 확인 결과**
- 이번에 요청할 주소는 두 종류입니다: 월별 목록 `/product/category/BestSellerContents`, 도서 상세 `/product/goods/{상품번호}`.
- `Disallow` 목록에는 `/goods/`(루트 바로 아래), `/product/goods/addModules/`, `/product/goods/communityModules/`, `/product/goods/JsonGoodsList` 등이 있지만 위 두 주소는 막혀 있지 않습니다.
- 따라서 요청 간격(1초)을 지키며 필요한 범위만 수집합니다.

# YES24 월별 베스트셀러 수집 (2023.01 ~ 2025.12, 월별 1~20위)

## 1. 월별 베스트셀러 페이지의 주소 규칙

브라우저에서 보는 월별 베스트셀러 페이지는 다음 주소입니다.

```
https://www.yes24.com/product/category/monthweekbestseller?categoryNumber=001&type=month&saleYear=2023&saleMonth=1
```

그런데 이 페이지를 `requests`로 받아 보면 **`saleYear`/`saleMonth` 값과 상관없이 항상 '현재' 순위 24권**이 들어 있습니다.
실제 연·월별 목록은 페이지가 열린 뒤 아래 주소를 AJAX로 호출해서 채워 넣습니다.

```
https://www.yes24.com/product/category/BestSellerContents
    ?categoryNumber=001            # 001 = 국내도서 종합
    &bestType=MONTH_BESTSELLER&type=month
    &saleYear=2023&saleMonth=1     # ← 연도·월 파라미터
    &pageNumber=1&pageSize=20      # ← 1페이지에 20권 = 1~20위만
    &sumGb=06&goodsStatGb=06 ...   # (사이트 기본값 그대로)
```

- `saleYear`, `saleMonth`만 바꾸면 원하는 달의 순위를 받을 수 있습니다.
- `pageSize=20`, `pageNumber=1`로 고정해 **1~20위만** 요청하고, 2페이지 이후는 요청하지 않습니다.

**requests + BeautifulSoup을 쓰는 이유**: AJAX 주소의 응답은 JavaScript 실행이 필요 없는 **완성된 HTML 조각**이므로
Selenium 없이 `requests`로 직접 호출하고 `BeautifulSoup`으로 파싱할 수 있습니다.

## 2. 분야 정보
- 목록 응답에는 분야(카테고리)가 없습니다.
- 그래서 각 책의 상세 페이지(`/product/goods/{상품번호}`)의 **'카테고리 분류'** 경로
  (예: `국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설`) 중 **두 번째 단계(대분류)**를 분야로 씁니다.
- 경로가 여러 개면 첫 번째 경로(대표 분류)를 사용합니다.
- 같은 책은 상세 페이지를 한 번만 요청하도록 `category_cache`에 저장해 재사용합니다.

In [2]:
import re
import time

import pandas as pd
from bs4 import BeautifulSoup

LIST_URL = "https://www.yes24.com/product/category/BestSellerContents"
GOODS_URL = "https://www.yes24.com/product/goods/{}"
TOP_N = 20                                                    # 매월 1~20위만
MONTHS = [(y, m) for y in (2023, 2024, 2025) for m in range(1, 13)]   # 36개월
DELAY = 1                                                     # 요청마다 1초 대기

session = requests.Session()
session.headers.update(HEADERS)

category_cache = {}   # 상품번호 -> (분야, 카테고리경로) : 같은 책은 한 번만 요청
month_cache = {}      # (연도, 월) -> 수집 결과 : 테스트한 달은 다시 요청하지 않음


def text_of(node):
    return re.sub(r"\s+", " ", node.get_text(" ", strip=True)).strip() if node else ""


def get_season(month):
    if month in (3, 4, 5):
        return "봄"
    if month in (6, 7, 8):
        return "여름"
    if month in (9, 10, 11):
        return "가을"
    return "겨울"   # 12, 1, 2월


def fetch_month_list(year, month):
    """월별 베스트셀러 1~20위 목록 (분야 제외)"""
    params = {
        "categoryNumber": "001", "sumGb": "06", "sex": "A", "age": 255,
        "goodsTp": 0, "addOptionTp": 0, "excludeTp": 2,
        "pageNumber": 1, "pageSize": TOP_N, "goodsStatGb": "06", "eBookTp": 0,
        "bestType": "MONTH_BESTSELLER", "type": "month",
        "saleYear": year, "saleMonth": month,
    }
    r = session.get(LIST_URL, params=params, timeout=30)
    r.raise_for_status()
    time.sleep(DELAY)
    soup = BeautifulSoup(r.text, "html.parser")

    books = []
    for li in soup.select("#yesBestList > li[data-goods-no]"):
        name = li.select_one("a.gd_name")
        if name is None:
            continue
        auth = li.select_one(".info_auth")
        if auth is not None:
            for more in auth.select(".moreAuthArea"):   # '정보 더 보기' 펼침 목록(중복 이름) 제외
                more.decompose()
        books.append({
            "연도": year,
            "월": month,
            "순위": text_of(li.select_one("em.ico.rank")),
            "상품번호": li["data-goods-no"],
            "제목": text_of(name),
            "저자": text_of(auth),
            "출판사": text_of(li.select_one(".info_pub")),
        })
    return books[:TOP_N]


def fetch_category(goods_no):
    """상세 페이지 '카테고리 분류'의 첫 번째 경로에서 두 번째 단계(대분류)를 반환"""
    if goods_no in category_cache:          # 이미 가져온 책이면 재사용
        return category_cache[goods_no]
    try:
        r = session.get(GOODS_URL.format(goods_no), timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        paths = []
        for dl in soup.select("#infoset_goodsCate dl"):
            if text_of(dl.find("dt")) != "카테고리 분류":
                continue
            for li in dl.select("ul.yesAlertLi > li"):
                parts = [text_of(a) for a in li.find_all("a")]
                if parts:
                    paths.append(" > ".join(parts))
        first = paths[0].split(" > ") if paths else []
        result = (first[1] if len(first) > 1 else None, paths[0] if paths else None)
    except requests.RequestException as e:
        print(f"  [경고] 상품번호 {goods_no} 상세 페이지 실패: {e}")
        result = (None, None)
    time.sleep(DELAY)
    category_cache[goods_no] = result
    return result


def scrape_month(year, month):
    """한 달치: 목록 + 분야 + 계절"""
    if (year, month) in month_cache:
        books = month_cache[(year, month)]
    else:
        books = fetch_month_list(year, month)
        for b in books:
            b["분야"], b["카테고리경로"] = fetch_category(b["상품번호"])
            b["계절"] = get_season(month)
        month_cache[(year, month)] = books
    print(f"[{year}년 {month}월] {len(books)}권 수집")
    return books

## 3. 테스트 수집: 2023년 1월 한 달치
전체 36개월을 돌리기 전에 2023년 1월만 먼저 수집해 순위·제목·저자·출판사·분야가 제대로 들어오는지 확인합니다.
(이 결과는 `month_cache`에 저장되어 전체 수집 때 다시 요청하지 않습니다.)

In [3]:
test = pd.DataFrame(scrape_month(2023, 1))
print("상세 페이지 요청한 책 수:", len(category_cache))
test

[2023년 1월] 20권 수집
상세 페이지 요청한 책 수: 20


,연도,월,순위,상품번호,제목,저자,출판사,분야,카테고리경로,계절
0,2023,1,1,115142458,만일 내가 인생을 다시 산다면 (35만 부 기념 스페셜 에디션),김혜남 저,메이븐,인문,국내도서 > 인문 > 주제로 읽는 인문학 > 노년 / 죽음,겨울
1,2023,1,2,116579452,스카이 버스,분당강쌤 저,다산에듀,가정 살림,국내도서 > 가정 살림 > 자녀교육 > 학습법일반,겨울
2,2023,1,3,116046489,생에 감사해,김혜자 저,수오서재,에세이,국내도서 > 에세이 > 명사/연예인 에세이,겨울
3,2023,1,4,99308021,불편한 편의점,김호연 저,나무옆의자,소설/시/희곡,국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설,겨울
4,2023,1,5,111088149,불편한 편의점 2,김호연 저,나무옆의자,소설/시/희곡,국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설,겨울
5,2023,1,6,112253263,아버지의 해방일지,정지아 저,창비,소설/시/희곡,국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설,겨울
6,2023,1,7,9349031,원씽 THE ONE THING,"게리 켈러 , 제이 파파산 저/ 구세희 역",비즈니스북스,자기계발,국내도서 > 자기계발 > 처세술/삶의 자세,겨울
7,2023,1,8,116206629,"2023 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상",최태성 저,이투스북,수험서 자격증,"국내도서 > 수험서 자격증 > 한국사능력검정시험 > 심화(1,2,3급)",겨울
8,2023,1,9,113416767,트렌드 코리아 2023,"김난도 , 전미영 , 최지혜 , 이수진 , 권정윤 저 외 5명",미래의창,경제 경영,국내도서 > 경제 경영 > 경제/경제학 > 각국 경제/경제사/전망 > 경제전망,겨울
9,2023,1,10,116789002,미스터 프레지던트,탁현민 저,메디치미디어,사회 정치,국내도서 > 사회 정치 > 언론학/미디어론 > 신문/방송,겨울


**테스트 결과 확인**
- 2023년 1월 1~20위 20권이 모두 수집되었고, 순위·제목·저자·출판사·분야·카테고리경로가 빈 값 없이 들어왔습니다.
- 1위는 '만일 내가 인생을 다시 산다면'(인문), 2위 '스카이 버스'(가정 살림)이며, 분야는 상세 페이지 경로의 두 번째 단계(예: `국내도서 > 소설/시/희곡 > 한국소설` → 소설/시/희곡)로 정상 추출되었습니다.
- 상세 페이지는 20번 요청했고(서로 다른 책 20권), 파싱 규칙이 맞으므로 전체 수집을 진행합니다.

## 4. 전체 수집: 2023년 1월 ~ 2025년 12월 (36개월 × 20권)

In [4]:
rows = []
for year, month in MONTHS:
    rows.extend(scrape_month(year, month))

print()
print("수집 행 수:", len(rows))
print("상세 페이지를 요청한 서로 다른 책 수:", len(category_cache))

[2023년 1월] 20권 수집


[2023년 2월] 20권 수집


[2023년 3월] 20권 수집


[2023년 4월] 20권 수집


[2023년 5월] 20권 수집


...

[2025년 8월] 20권 수집


[2025년 9월] 20권 수집


[2025년 10월] 20권 수집


[2025년 11월] 20권 수집


[2025년 12월] 20권 수집

수집 행 수: 720
상세 페이지를 요청한 서로 다른 책 수: 304


## 5. 원본 저장 (가공 없음)

In [5]:
raw = pd.DataFrame(rows, columns=["연도", "월", "계절", "순위", "상품번호", "제목", "저자", "출판사", "분야", "카테고리경로"])
raw.to_csv("yes24_seasonal_raw.csv", index=False, encoding="utf-8-sig")
print(raw.shape)
raw.head(10)

(720, 10)


,연도,월,계절,순위,상품번호,제목,저자,출판사,분야,카테고리경로
0,2023,1,겨울,1,115142458,만일 내가 인생을 다시 산다면 (35만 부 기념 스페셜 에디션),김혜남 저,메이븐,인문,국내도서 > 인문 > 주제로 읽는 인문학 > 노년 / 죽음
1,2023,1,겨울,2,116579452,스카이 버스,분당강쌤 저,다산에듀,가정 살림,국내도서 > 가정 살림 > 자녀교육 > 학습법일반
2,2023,1,겨울,3,116046489,생에 감사해,김혜자 저,수오서재,에세이,국내도서 > 에세이 > 명사/연예인 에세이
3,2023,1,겨울,4,99308021,불편한 편의점,김호연 저,나무옆의자,소설/시/희곡,국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설
4,2023,1,겨울,5,111088149,불편한 편의점 2,김호연 저,나무옆의자,소설/시/희곡,국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설
5,2023,1,겨울,6,112253263,아버지의 해방일지,정지아 저,창비,소설/시/희곡,국내도서 > 소설/시/희곡 > 한국소설 > 한국 장편소설
6,2023,1,겨울,7,9349031,원씽 THE ONE THING,"게리 켈러 , 제이 파파산 저/ 구세희 역",비즈니스북스,자기계발,국내도서 > 자기계발 > 처세술/삶의 자세
7,2023,1,겨울,8,116206629,"2023 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상",최태성 저,이투스북,수험서 자격증,"국내도서 > 수험서 자격증 > 한국사능력검정시험 > 심화(1,2,3급)"
8,2023,1,겨울,9,113416767,트렌드 코리아 2023,"김난도 , 전미영 , 최지혜 , 이수진 , 권정윤 저 외 5명",미래의창,경제 경영,국내도서 > 경제 경영 > 경제/경제학 > 각국 경제/경제사/전망 > 경제전망
9,2023,1,겨울,10,116789002,미스터 프레지던트,탁현민 저,메디치미디어,사회 정치,국내도서 > 사회 정치 > 언론학/미디어론 > 신문/방송


**수집 결과**
- 36개월 모두 `[연도년 월월] 20권 수집`으로, 총 720행(36개월 × 20권)이 수집되었습니다. 2023년 1월은 테스트 결과를 재사용해 다시 요청하지 않았습니다.
- 상세 페이지는 서로 다른 책 304권에 대해 한 번씩만 요청했습니다(재등장한 책은 `category_cache` 재사용).
- 가공 없이 `yes24_seasonal_raw.csv`(720행 × 10열, utf-8-sig)로 저장했습니다. 정리는 `02_yes24_EDA_REPORT.ipynb`에서 합니다.